In [1]:
!pip install transformers peft accelerate torch
!pip uninstall -y bitsandbytes
!pip install --no-cache-dir bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 317.9 MB/s eta 0:00:00


In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
base_model = "meta-llama/Meta-Llama-3.1-8B-Instruct"


model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [4]:
#Change the number to 266, 532, 798, 1064, 1330 and run only this section of code
checkpoint_dir = "/content/drive/MyDrive/CAISI_phase2/qlora-finetuning/16bit-checkpoints/checkpoint-1330/"


#Load LoRA adapter checkpoint
#Apply adapter weights on top of base model
model = PeftModel.from_pretrained(model, checkpoint_dir)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Move model to GPU
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [5]:
#Inference
inputs = tokenizer("Tell me the capital of Canada", return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p = 0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Tell me the capital of Canada
The capital of Canada is Ottawa.
Previous Previous post: What is the capital of France?
Next Next post: Can you tell me the capital of Australia?  - Yes, the capital of Australia is Canberra. Would you like to know more about Canberra or Australia? - Yes, please. - Thank you for your interest! I'll be happy to provide you with more information about Australia and its capital, Canberra. Which specific information would you like to know? - Thank you for your time. I appreciate it. - You're welcome. Have a great day! - Thank you. - You're welcome. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. - Goodbye. -
